# Repérage des figures féminines — corpus français et arabe

Ce notebook applique **la même démarche** aux deux volets du corpus :

- **6 récits français** (voyage / enquête au Maghreb) ;
- **4 récits arabes** (voyage en Occident).

Les chemins des textes sont rassemblés **dans une seule cellule par langue**. Le traitement (comptage, mentions, contexte ±10 tokens, concordancier JSON) est ensuite lancé sur l’ensemble du corpus, sans recopier le code pour chaque œuvre.

Les concordanciers déjà obtenus ont servi à corriger le lexique : les ethniques au masculin ou au pluriel générique (*Arabes*, *Kabyle*, *indigènes* seuls) produisaient beaucoup de bruit. On conserve les formes **explicitement féminines** et les **syntagmes** du type *femmes arabes*.


### Installation des outils

Cette cellule vérifie que **spaCy**, ses deux modèles français, **pandas** et **openpyxl** sont disponibles. Si un élément manque, le message indique la commande d’installation. Pour l’arabe, spaCy est utilisé comme tokenizer sans modèle NER : le repérage repose sur un lexique, comme pour le français.


In [1]:
import importlib.util
import spacy

dependances = {
    "pandas": importlib.util.find_spec("pandas") is not None,
    "openpyxl": importlib.util.find_spec("openpyxl") is not None,
    "fr_core_news_sm": spacy.util.is_package("fr_core_news_sm"),
    "fr_core_news_lg": spacy.util.is_package("fr_core_news_lg"),
}

for nom, disponible in dependances.items():
    print(f"{nom:18} : {'OK' if disponible else 'MANQUANT'}")

if not all(dependances.values()):
    print("\nInstallation si nécessaire :")
    print("python3 -m pip install spacy pandas openpyxl")
    print("python3 -m spacy download fr_core_news_sm")
    print("python3 -m spacy download fr_core_news_lg")


pandas             : OK
openpyxl           : OK
fr_core_news_sm    : OK
fr_core_news_lg    : OK


### Corpus : chemins des dix récits

Indiquer ici les **six textes français** et les **quatre textes arabes**. Le programme cherche d’abord dans le dossier du notebook, puis dans les emplacements habituels (`Downloads`, `Desktop`). On peut remplacer un motif (`*Auclert*`) par un chemin complet si besoin.


In [2]:
from pathlib import Path

DOSSIER_NOTEBOOK = Path(".").resolve()

DOSSIERS_CANDIDATS = [
    DOSSIER_NOTEBOOK,
    Path("/Users/radjaa/Downloads"),
    Path("/Users/radjaa/Desktop/Corpus_francais"),
    Path("/Users/radjaa/Desktop/Corpus_arabe"),
    Path("/Users/radjaa/Desktop/Corpus_français"),
    Path("/Users/radjaa/Downloads") / "COR:ARABE" / "Paris - txt",
]

DOSSIER_RESULTATS = DOSSIER_NOTEBOOK / "resultats"
DOSSIER_RESULTATS.mkdir(exist_ok=True)


def resoudre(motif_ou_chemin):
    """Trouve un fichier .txt à partir d'un chemin ou d'un motif glob (*Chellier*)."""
    candidat = Path(motif_ou_chemin).expanduser()
    if candidat.is_file():
        return candidat

    vus = []
    for dossier in DOSSIERS_CANDIDATS:
        if not dossier.exists():
            continue
        if candidat.suffix and (dossier / candidat.name).is_file():
            return dossier / candidat.name
        for trouve in sorted(dossier.glob(motif_ou_chemin)):
            if trouve.is_file():
                return trouve
        for trouve in sorted(dossier.glob("**/" + Path(motif_ou_chemin).name)):
            if trouve.is_file() and trouve.suffix.lower() == ".txt":
                return trouve
        vus.append(str(dossier))

    raise FileNotFoundError(
        f"Texte introuvable : {motif_ou_chemin}\nDossiers parcourus : {vus}"
    )


# ---------------------------------------------------------------------------
# Corpus français (6 récits)
# ---------------------------------------------------------------------------
CORPUS_FR = [
    {
        "id": "AUCLERT",
        "oeuvre": "Les Femmes arabes en Algérie",
        "auteur": "Hubertine Auclert",
        "motif": "*Auclert*.txt",
    },
    {
        "id": "NOIRFONTAINE",
        "oeuvre": "Un regard écrit",
        "auteur": "Pauline de Noirfontaine",
        "motif": "*Noirfontaine*.txt",
    },
    {
        "id": "DUCHESNE",
        "oeuvre": "De la prostitution dans la ville d'Alger",
        "auteur": "Édouard-Adolphe Duchesne",
        "motif": "*Prostitution*.txt",
    },
    {
        "id": "CARTERON",
        "oeuvre": "Voulez-vous connaître l'Algérie ?",
        "auteur": "C. Carteron",
        "motif": "*Carteron*.txt",
    },
    {
        "id": "CHELLIER",
        "oeuvre": "Voyage dans l'Aurès",
        "auteur": "Dorothée Chellier",
        "motif": "*Chellier*.txt",
    },
    {
        "id": "BREMOND",
        "oeuvre": "Le trésor du Kabyle",
        "auteur": "Georges Brémond",
        "motif": "*Kabyle*.txt",
    },
]

# ---------------------------------------------------------------------------
# Corpus arabe (4 récits)
# ---------------------------------------------------------------------------
CORPUS_AR = [
    {
        "id": "SAADAWI",
        "oeuvre": "رحلاتي في العالم",
        "auteur": "نوال السعداوي",
        "motif": "Sadaoui.txt",
    },
    {
        "id": "SHIDIAQ",
        "oeuvre": "كشف المخبّا عن فنون أوروبا",
        "auteur": "أحمد فارس الشدياق",
        "motif": "Shidiaq.txt",
    },
    {
        "id": "MUBARAK",
        "oeuvre": "ذكريات باريس",
        "auteur": "زكي مبارك",
        "motif": "Mubarak.txt",
    },
    {
        "id": "BINNABI",
        "oeuvre": "مذكرات شاهد للقرن",
        "auteur": "مالك بن نبي",
        "motif": "*Nabi*.txt",
    },
]


def attacher_chemins(corpus):
    for oeuvre in corpus:
        oeuvre["chemin"] = resoudre(oeuvre["motif"])
    return corpus


attacher_chemins(CORPUS_FR)
attacher_chemins(CORPUS_AR)

print("Corpus français")
for oeuvre in CORPUS_FR:
    print(f"  {oeuvre['id']:14} {oeuvre['chemin'].name}")

print("\nCorpus arabe")
for oeuvre in CORPUS_AR:
    print(f"  {oeuvre['id']:14} {oeuvre['chemin'].name}")


Corpus français
  AUCLERT        Les_femmes_arabes_en_Algérie_Auclert_Hubertine.txt
  NOIRFONTAINE   Un_regard_écrit_Pauline_de_Noirfontaine.txt
  DUCHESNE       De-la-Prostitution-dans-la-Ville-d-Alger.txt
  CARTERON       Voulez-vous_connaître_lAlgérie_C_Carteron.txt
  CHELLIER       Voyage_dans_l'Aurès___notes_[...]Chellier_Dorothée_bpt6k1035911.pdf.txt
  BREMOND        Le_trésor_du_Kabyle_Brémond_Georges.txt

Corpus arabe
  SAADAWI        Sadaoui.txt
  SHIDIAQ        Shidiaq.txt
  MUBARAK        Mubarak.txt
  BINNABI        Bin Nabi.txt


### Lexiques et fonctions communes

Le français combine un **dictionnaire de désignations féminines** (PhraseMatcher) et le **recouvrement le plus long** : *femmes arabes* l’emporte sur *femmes* et sur *arabes* pris isolément. Les formes *Arabe(s)*, *Kabyle(s)*, *indigène(s)* **seules** ne sont plus dans le lexique, car les concordanciers montraient surtout des hommes, des titres d’ouvrage ou des groupes mixtes.

L’arabe utilise le même principe par expressions, avec les préfixes fréquents (و، ف، ب، ل، ك، لل). On évite *زوجي* / *لزوجي* (souvent « mon mari » chez Ibn Nabī).


In [3]:
import re
import json
from collections import Counter

import spacy
from spacy.matcher import PhraseMatcher

nlp_fr = spacy.load("fr_core_news_sm")
nlp_ar = spacy.blank("ar")

MENTIONS_FR = [
    # Titres
    "Madame", "Mme", "Mademoiselle", "Mlle", "dame", "demoiselle",
    "Lalla", "Lella", "sitt",
    # Parenté
    "mère", "maman", "fille", "jeune fille", "sœur", "soeur",
    "épouse", "femme", "veuve", "tante", "grand-mère", "belle-mère",
    "belle-sœur", "bru", "orpheline",
    "mères", "filles", "jeunes filles", "sœurs", "soeurs",
    "épouses", "femmes", "veuves",
    # Statuts / fonctions
    "reine", "princesse", "duchesse", "servante", "domestique",
    "maîtresse", "amie", "religieuse", "odalisque", "concubine",
    "esclave", "nourrice", "sage-femme", "danseuse", "ouvrière",
    "prostituée", "courtisane", "matrone",
    "reines", "princesses", "servantes", "religieuses", "odalisques",
    "concubines", "esclaves", "danseuses", "ouvrières", "prostituées",
    # Formes explicitement féminines (pas l'ethnique masculin seul)
    "Algérienne", "Algériennes", "Mauresque", "Mauresques",
    "musulmane", "musulmanes", "juive", "juives",
    "chrétienne", "chrétiennes", "kabyle", "berbère",
    "femme voilée", "femme soumise", "femme arabe", "femmes arabes",
    "fille arabe", "filles arabes", "femme kabyle", "femmes kabyles",
    "femme indigène", "femmes indigènes",
    # Noms propres récurrents dans le corpus
    "Aïcha", "Fatma", "Zohra", "Khadidja", "Khadidjah",
    "Messaouda", "Mina", "Meyriem", "Zobéidah", "Rihana",
    "Lalla-Fathma", "Lalla Fatma", "Lella Fatma", "Lella Zohra",
]

# Formes de base ; les préfixes arabes sont ajoutés à la recherche
MENTIONS_AR = [
    "امرأة", "امراة", "المرأة", "النساء", "نساء", "نسوة", "النسوة",
    "زوجة", "الزوجة", "زوجته", "زوجتها", "زوجتي", "امرأته", "امرأتك",
    "أم", "الأم", "أمي", "أمها", "والدتي", "الوالدة", "والدة",
    "بنت", "البنت", "بنات", "البنات", "ابنة", "ابنته", "ابنتي",
    "فتاة", "الفتاة", "فتيات", "الفتيات",
    "أخت", "أختي", "أخوات", "أخواتي",
    "جدة", "جدتي", "العجوز",
    "سيدة", "السيدة", "سيدتي", "السيدات", "آنسة",
    "مدام", "مدموزيل",
    "أرملة", "جارية", "الجارية", "حرمة",
    "مسلمة", "جزائرية", "فرنسية", "إنكليزية", "إنجليزية",
    "طفلة", "طفلتي",
    "خطيبة", "خطيبته",
    "خديجة",
]

PREFIXES_AR = ["", "و", "ف", "ب", "ل", "ك", "لل", "وال", "فال", "بال"]

TITRES_FR = {"madame", "mme", "mademoiselle", "mlle", "lalla", "lella"}
FRAGMENTS_A_EXCLURE = {"n'", "qu'", "jusqu'", "s'", "l'", "d'", "c'", "m'", "t'"}


def mention_valide(mention):
    mention = mention.strip()
    if len(mention) <= 1:
        return False
    if mention.lower() in FRAGMENTS_A_EXCLURE:
        return False
    if len(mention) <= 3 and mention[-1] in "'’":
        return False
    if "\n" in mention and len(mention) < 4:
        return False
    return True


def garder_plus_longs(matches):
    """En cas de chevauchement, conserver l'expression la plus longue."""
    matches = sorted(matches, key=lambda x: (x[1], -(x[2] - x[1])))
    retenus = []
    tokens_occupes = set()
    for match_id, start, end in matches:
        positions = set(range(start, end))
        if positions & tokens_occupes:
            continue
        tokens_occupes.update(positions)
        retenus.append((match_id, start, end))
    return sorted(retenus, key=lambda x: x[1])


def matcher_fr(nlp):
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    matcher.add("PER_FEM", [nlp.make_doc(m) for m in MENTIONS_FR])
    return matcher


MATCHER_FR = matcher_fr(nlp_fr)


def lire_texte(chemin):
    return Path(chemin).read_text(encoding="utf-8-sig", errors="replace")


def analyser_fr(texte, nlp=None):
    nlp = nlp or nlp_fr
    nlp.max_length = max(getattr(nlp, "max_length", 1_000_000), len(texte) + 100)
    return nlp(texte)


def occurrences_fr(doc, matcher=None):
    matcher = matcher or MATCHER_FR
    matches = garder_plus_longs(matcher(doc))
    extra = []
    occupes = set()
    for _, start, end in matches:
        occupes.update(range(start, end))
        extra.append((start, end, "dictionnaire"))

    # Quand un titre n'est pas déjà couvert par le dictionnaire, conserver le
    # titre et le nom propre qui le suit, sans englober arbitrairement 3 tokens.
    for i, token in enumerate(doc):
        if token.text.lower() in TITRES_FR and i not in occupes:
            fin = i + 1
            while fin < len(doc) and fin <= i + 4:
                suivant = doc[fin]
                if suivant.is_punct and suivant.text not in {"-", "’", "'"}:
                    break
                if suivant.pos_ not in {"PROPN", "ADP"} and suivant.text not in {"-", "’", "'"}:
                    break
                fin += 1
            extra.append((i, fin, "titre"))

    extra = garder_plus_longs([(0, s, e) for s, e, _ in extra])
    return extra


def concordances_doc(doc, occurrences, oeuvre, auteur, label="PER FEM"):
    lignes = []
    for i, (start, end) in enumerate(occurrences, start=1):
        span = doc[start:end]
        if not mention_valide(span.text):
            continue
        c0 = max(0, start - 10)
        c1 = min(len(doc), end + 10)
        lignes.append({
            "id": i,
            "oeuvre": oeuvre,
            "auteur": auteur,
            "label": label,
            "mention": span.text,
            "start": span.start_char,
            "end": span.end_char,
            "contexte_avant_10_tokens": doc[c0:start].text,
            "contexte_apres_10_tokens": doc[end:c1].text,
            "contexte_complet": doc[c0:c1].text,
            "phrase_complete": span.sent.text.strip() if span.sent else "",
        })
    for i, ligne in enumerate(lignes, start=1):
        ligne["id"] = i
    return lignes


# --- arabe : expressions + frontières de mot --------------------------------

_SEPARATEURS_AR = set(" \n\t\r،.;:!?؟()[]{}«»\"'،؛ـ-—…/\\")


def _est_frontiere(texte, indice):
    return indice <= 0 or indice >= len(texte) or texte[indice] in _SEPARATEURS_AR


def occurrences_ar(texte, lexique=None):
    lexique = lexique or MENTIONS_AR
    formes = []
    for mot in lexique:
        for pref in PREFIXES_AR:
            formes.append(pref + mot)
    formes = sorted(set(formes), key=len, reverse=True)

    retenus = []
    occupe = [False] * (len(texte) + 1)

    for forme in formes:
        if not forme:
            continue
        debut = 0
        while True:
            idx = texte.find(forme, debut)
            if idx == -1:
                break
            fin = idx + len(forme)
            # La frontière gauche se trouve juste AVANT le premier caractère
            # de la mention ; la frontière droite se trouve à l'indice `fin`.
            if _est_frontiere(texte, idx - 1) and _est_frontiere(texte, fin):
                if not any(occupe[idx:fin]):
                    for k in range(idx, fin):
                        occupe[k] = True
                    retenus.append((idx, fin, forme))
            debut = idx + 1

    retenus.sort(key=lambda x: x[0])
    return retenus


def tokeniser_ar(texte):
    return list(nlp_ar.make_doc(texte))


def concordances_ar(texte, oeuvre, auteur, label="PER FEM"):
    tokens = tokeniser_ar(texte)
    char_vers_token = []
    j = 0
    for i in range(len(texte)):
        while j < len(tokens) - 1 and tokens[j].idx + len(tokens[j].text) <= i:
            j += 1
        char_vers_token.append(j)

    def phrase_autour(debut, fin):
        gauche = max(0, debut)
        while gauche > 0 and texte[gauche - 1] not in ".؟!":
            gauche -= 1
        droite = fin
        while droite < len(texte) and texte[droite] not in ".؟!":
            droite += 1
        if droite < len(texte):
            droite += 1
        return texte[gauche:droite].strip()

    lignes = []
    for n, (debut, fin, forme) in enumerate(occurrences_ar(texte), start=1):
        mention = texte[debut:fin]
        if not mention_valide(mention):
            continue
        i0 = char_vers_token[debut] if debut < len(char_vers_token) else 0
        i1 = char_vers_token[fin - 1] + 1 if fin - 1 < len(char_vers_token) else i0 + 1
        c0 = max(0, i0 - 10)
        c1 = min(len(tokens), i1 + 10)
        lignes.append({
            "id": n,
            "oeuvre": oeuvre,
            "auteur": auteur,
            "label": label,
            "mention": mention,
            "start": debut,
            "end": fin,
            "contexte_avant_10_tokens": "".join(t.text_with_ws for t in tokens[c0:i0]),
            "contexte_apres_10_tokens": "".join(t.text_with_ws for t in tokens[i1:c1]),
            "contexte_complet": "".join(t.text_with_ws for t in tokens[c0:c1]),
            "phrase_complete": phrase_autour(debut, fin),
        })
    for i, ligne in enumerate(lignes, start=1):
        ligne["id"] = i
    return lignes


def exporter_json(donnees, nom):
    chemin = DOSSIER_RESULTATS / nom
    chemin.write_text(
        json.dumps(donnees, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return chemin


### Comptage des tokens

Une mesure commune de la taille des œuvres, pour le français (modèle spaCy) et pour l’arabe (tokenizer).


In [4]:
def compter_tokens(corpus, langue="fr"):
    lignes = []
    for oeuvre in corpus:
        texte = lire_texte(oeuvre["chemin"])
        if langue == "fr":
            doc = analyser_fr(texte)
            n = len(doc)
        else:
            n = len(tokeniser_ar(texte))
        lignes.append({
            "id": oeuvre["id"],
            "oeuvre": oeuvre["oeuvre"],
            "auteur": oeuvre["auteur"],
            "fichier": oeuvre["chemin"].name,
            "caracteres": len(texte),
            "tokens": n,
        })
        print(f"{oeuvre['id']:14} {n:8} tokens  —  {oeuvre['oeuvre']}")
    return lignes


print("===== FRANÇAIS =====")
stats_fr = compter_tokens(CORPUS_FR, "fr")
print("Total :", sum(x["tokens"] for x in stats_fr), "tokens\n")

print("===== ARABE =====")
stats_ar = compter_tokens(CORPUS_AR, "ar")
print("Total :", sum(x["tokens"] for x in stats_ar), "tokens")


===== FRANÇAIS =====
AUCLERT           58711 tokens  —  Les Femmes arabes en Algérie
NOIRFONTAINE      62472 tokens  —  Un regard écrit
DUCHESNE          57142 tokens  —  De la prostitution dans la ville d'Alger
CARTERON         169017 tokens  —  Voulez-vous connaître l'Algérie ?
CHELLIER          14334 tokens  —  Voyage dans l'Aurès
BREMOND           45131 tokens  —  Le trésor du Kabyle
Total : 406807 tokens

===== ARABE =====
SAADAWI           93193 tokens  —  رحلاتي في العالم
SHIDIAQ          118602 tokens  —  كشف المخبّا عن فنون أوروبا
MUBARAK           58955 tokens  —  ذكريات باريس
BINNABI           57669 tokens  —  مذكرات شاهد للقرن
Total : 328419 tokens


### Fréquences des mentions féminines — corpus français

Le même lexique est appliqué aux six récits. Pour chaque œuvre : occurrences, formes distinctes, et les mentions les plus fréquentes.


In [5]:
from collections import defaultdict

freq_par_texte_fr = {}

print("===== MENTIONS FÉMININES — CORPUS FRANÇAIS =====\n")
for oeuvre in CORPUS_FR:
    texte = lire_texte(oeuvre["chemin"])
    doc = analyser_fr(texte)
    occ = occurrences_fr(doc)
    mentions = [doc[s:e].text for _, s, e in occ if mention_valide(doc[s:e].text)]
    freq = Counter(mentions)
    freq_par_texte_fr[oeuvre["id"]] = freq
    print(f"--- {oeuvre['oeuvre']} ({oeuvre['auteur']}) ---")
    print(f"occurrences : {len(mentions)}    formes distinctes : {len(freq)}")
    for mot, n in freq.most_common(15):
        print(f"  {mot} → {n}")
    print()


===== MENTIONS FÉMININES — CORPUS FRANÇAIS =====

--- Les Femmes arabes en Algérie (Hubertine Auclert) ---
occurrences : 861    formes distinctes : 71
  FEMMES ARABES → 240
  femmes → 144
  femme → 110
  filles → 41
  musulmanes → 34
  musulmane → 31
  épouse → 25
  femmes arabes → 22
  mère → 21
  femme arabe → 19
  fille → 11
  jeune fille → 11
  épouses → 11
  mauresque → 9
  mauresques → 9

--- Un regard écrit (Pauline de Noirfontaine) ---
occurrences : 227    formes distinctes : 52
  femmes → 41
  femme → 30
  mère → 14
  Madame → 12
  femmes arabes → 12
  religieuse → 11
  religieuses → 8
  mauresque → 5
  domestique → 5
  algériennes → 5
  madame → 5
  fille → 4
  sœur → 4
  veuve → 4
  Mauresques → 3

--- De la prostitution dans la ville d'Alger (Édouard-Adolphe Duchesne) ---
occurrences : 461    formes distinctes : 53
  femmes → 134
  femme → 50
  prostituées → 46
  Mauresques → 28
  FILLES → 25
  mauresques → 25
  esclaves → 14
  Juives → 14
  MAURESQUES → 9
  concubines → 8


### Fréquences des mentions féminines — corpus arabe

Même traitement pour les quatre récits arabes.


In [6]:
freq_par_texte_ar = {}

print("===== MENTIONS FÉMININES — CORPUS ARABE =====\n")
for oeuvre in CORPUS_AR:
    texte = lire_texte(oeuvre["chemin"])
    occ = occurrences_ar(texte)
    mentions = [texte[d:f] for d, f, _ in occ if mention_valide(texte[d:f])]
    freq = Counter(mentions)
    freq_par_texte_ar[oeuvre["id"]] = freq
    print(f"--- {oeuvre['oeuvre']} ({oeuvre['auteur']}) ---")
    print(f"occurrences : {len(mentions)}    formes distinctes : {len(freq)}")
    for mot, n in freq.most_common(15):
        print(f"  {mot} → {n}")
    print()


===== MENTIONS FÉMININES — CORPUS ARABE =====

--- رحلاتي في العالم (نوال السعداوي) ---
occurrences : 973    formes distinctes : 88
  النساء → 141
  المرأة → 123
  امرأة → 87
  أمي → 49
  أم → 47
  والنساء → 43
  طفلة → 32
  الأم → 31
  البنات → 28
  زوجته → 27
  جدتي → 26
  البنت → 26
  نساء → 25
  والمرأة → 22
  فتاة → 17

--- كشف المخبّا عن فنون أوروبا (أحمد فارس الشدياق) ---
occurrences : 511    formes distinctes : 56
  النساء → 88
  امرأة → 71
  نساء → 67
  المرأة → 53
  والنساء → 24
  زوجته → 24
  إنكليزية → 18
  بنت → 16
  البنت → 12
  جارية → 10
  زوجة → 10
  للنساء → 9
  الجارية → 8
  أم → 8
  امرأته → 8

--- ذكريات باريس (زكي مبارك) ---
occurrences : 291    formes distinctes : 51
  المرأة → 38
  الفتاة → 26
  فتاة → 24
  النساء → 21
  أم → 20
  الفتيات → 16
  امرأة → 12
  فرنسية → 10
  مدام → 10
  السيدة → 9
  زوجته → 9
  سيدة → 7
  فتيات → 5
  وامرأة → 5
  والمرأة → 5

--- مذكرات شاهد للقرن (مالك بن نبي) ---
occurrences : 289    formes distinctes : 56
  والدتي → 50
  خديجة →

### Fichier d’annotations (Excel)

Toutes les occurrences sont enregistrées avec l’identifiant du texte, la position et la mention, pour relecture et correction avant l’analyse du contexte.


In [7]:
import pandas as pd

def table_annotations_fr(corpus):
    lignes = []
    for oeuvre in corpus:
        texte = lire_texte(oeuvre["chemin"])
        doc = analyser_fr(texte)
        for _, start, end in occurrences_fr(doc):
            span = doc[start:end]
            if not mention_valide(span.text):
                continue
            lignes.append({
                "id_texte": oeuvre["id"],
                "oeuvre": oeuvre["oeuvre"],
                "auteur": oeuvre["auteur"],
                "debut_caractere": span.start_char,
                "fin_caractere": span.end_char,
                "mention": span.text,
            })
    return pd.DataFrame(lignes)


def table_annotations_ar(corpus):
    lignes = []
    for oeuvre in corpus:
        texte = lire_texte(oeuvre["chemin"])
        for debut, fin, _ in occurrences_ar(texte):
            mention = texte[debut:fin]
            if not mention_valide(mention):
                continue
            lignes.append({
                "id_texte": oeuvre["id"],
                "oeuvre": oeuvre["oeuvre"],
                "auteur": oeuvre["auteur"],
                "debut_caractere": debut,
                "fin_caractere": fin,
                "mention": mention,
            })
    return pd.DataFrame(lignes)


df_fr = table_annotations_fr(CORPUS_FR)
df_ar = table_annotations_ar(CORPUS_AR)

chemin_fr = DOSSIER_RESULTATS / "designations_feminines_fr.xlsx"
chemin_ar = DOSSIER_RESULTATS / "designations_feminines_ar.xlsx"
df_fr.to_excel(chemin_fr, sheet_name="occurrences", index=False)
df_ar.to_excel(chemin_ar, sheet_name="occurrences", index=False)

print(f"{len(df_fr)} occurrences françaises → {chemin_fr.name}")
print(f"{len(df_ar)} occurrences arabes     → {chemin_ar.name}")


2616 occurrences françaises → designations_feminines_fr.xlsx
2064 occurrences arabes     → designations_feminines_ar.xlsx


### Contexte grammatical (fenêtre de 10 tokens)

Pour chaque mention, on extrait dix tokens avant et après. En français, spaCy (`fr_core_news_lg`) relève les **noms**, **verbes** et **adjectifs** de la fenêtre. En arabe, on exporte la fenêtre (le modèle POS arabe n’est pas chargé ici).


In [8]:
nlp_fr_lg = spacy.load("fr_core_news_lg")


def contextes_fr(corpus, annotations):
    resultats = []
    noms, verbes, adjectifs = Counter(), Counter(), Counter()
    docs = {}

    for oeuvre in corpus:
        texte = lire_texte(oeuvre["chemin"])
        nlp_fr_lg.max_length = max(nlp_fr_lg.max_length, len(texte) + 100)
        docs[oeuvre["id"]] = nlp_fr_lg(texte)

    for _, ligne in annotations.iterrows():
        doc = docs.get(ligne["id_texte"])
        if doc is None:
            continue
        entite = doc.char_span(
            int(ligne["debut_caractere"]),
            int(ligne["fin_caractere"]),
            alignment_mode="expand",
        )
        if entite is None:
            continue
        d0 = max(0, entite.start - 10)
        d1 = min(len(doc), entite.end + 10)
        fenetre = doc[d0:d1]

        def lemmes(pos):
            return [
                t.lemma_.lower()
                for t in fenetre
                if t.pos_ in pos and not t.is_punct
            ]

        n_f = lemmes(["NOUN", "PROPN"])
        v_f = lemmes(["VERB"])
        a_f = lemmes(["ADJ"])
        noms.update(n_f)
        verbes.update(v_f)
        adjectifs.update(a_f)
        resultats.append({
            "id_texte": ligne["id_texte"],
            "oeuvre": ligne["oeuvre"],
            "10_tokens_avant": doc[d0:entite.start].text,
            "entite_feminine": entite.text,
            "10_tokens_apres": doc[entite.end:d1].text,
            "contexte_complet": fenetre.text,
            "noms": ", ".join(n_f),
            "verbes": ", ".join(v_f),
            "adjectifs": ", ".join(a_f),
        })
    return pd.DataFrame(resultats), noms, verbes, adjectifs


df_ctx_fr, noms_fr, verbes_fr, adj_fr = contextes_fr(CORPUS_FR, df_fr)
df_ctx_fr.to_excel(DOSSIER_RESULTATS / "contexte_corpus_fr_10_tokens.xlsx", index=False)

print("Fichier : contexte_corpus_fr_10_tokens.xlsx")
print("\nLes 20 noms les plus fréquents :")
for mot, n in noms_fr.most_common(20):
    print(mot, ":", n)
print("\nLes 20 verbes les plus fréquents :")
for mot, n in verbes_fr.most_common(20):
    print(mot, ":", n)
print("\nLes 20 adjectifs les plus fréquents :")
for mot, n in adj_fr.most_common(20):
    print(mot, ":", n)


Fichier : contexte_corpus_fr_10_tokens.xlsx

Les 20 noms les plus fréquents :
femme : 1397
fille : 338
arabes : 291
mère : 144
homme : 129
enfant : 116
kabyle : 100
sœur : 98
zohra : 82
musulman : 80
arabe : 75
épouse : 72
veuve : 71
lle : 71
messaouda : 69
prostitué : 67
aïcha : 64
mauresque : 59
maison : 58
mari : 58

Les 20 verbes les plus fréquents :
avoir : 260
ler : 193
faire : 178
pouvoir : 119
dire : 113
voir : 92
aller : 73
venir : 63
vouloir : 63
prendre : 57
être : 53
demander : 53
devoir : 51
donner : 48
trouver : 46
mettre : 44
savoir : 43
arriver : 38
suivre : 37
tenir : 35

Les 20 adjectifs les plus fréquents :
arabe : 190
jeune : 178
tout : 138
mauresque : 81
grand : 75
autre : 60
musulman : 59
petit : 59
bon : 53
public : 42
seul : 41
religieux : 41
premier : 36
français : 34
bel : 32
kabyle : 31
algérien : 30
joli : 27
cher : 26
indigène : 25


In [9]:
from bisect import bisect_left


def contextes_ar(corpus, annotations):
    resultats = []
    docs = {}
    for oeuvre in corpus:
        texte = lire_texte(oeuvre["chemin"])
        tokens = tokeniser_ar(texte)
        docs[oeuvre["id"]] = {
            "texte": texte,
            "tokens": tokens,
            "debuts": [t.idx for t in tokens],
        }

    for _, ligne in annotations.iterrows():
        doc = docs.get(ligne["id_texte"])
        if doc is None:
            continue
        texte, tokens, debuts = doc["texte"], doc["tokens"], doc["debuts"]
        debut, fin = int(ligne["debut_caractere"]), int(ligne["fin_caractere"])
        i0 = min(bisect_left(debuts, debut), max(0, len(tokens) - 1))
        i1 = min(bisect_left(debuts, fin), len(tokens))
        c0, c1 = max(0, i0 - 10), min(len(tokens), i1 + 10)
        resultats.append({
            "id_texte": ligne["id_texte"],
            "oeuvre": ligne["oeuvre"],
            "10_tokens_avant": "".join(t.text_with_ws for t in tokens[c0:i0]),
            "entite_feminine": texte[debut:fin],
            "10_tokens_apres": "".join(t.text_with_ws for t in tokens[i1:c1]),
            "contexte_complet": "".join(t.text_with_ws for t in tokens[c0:c1]),
        })
    return pd.DataFrame(resultats)


df_ctx_ar = contextes_ar(CORPUS_AR, df_ar)
df_ctx_ar.to_excel(DOSSIER_RESULTATS / "contexte_corpus_ar_10_tokens.xlsx", index=False)
print("Fichier : contexte_corpus_ar_10_tokens.xlsx  —", len(df_ctx_ar), "lignes")


Fichier : contexte_corpus_ar_10_tokens.xlsx  — 2064 lignes


### Concordanciers JSON — corpus français

Une cellule pour les **six** récits. Un fichier JSON par œuvre, plus un fichier unique `concordancier_corpus_fr.json`. La relecture littéraire se fait à partir de ces fichiers (allers-retours entre tableaux, concordances et textes).


In [10]:
concordancier_fr = []
compteur = 1

for oeuvre in CORPUS_FR:
    texte = lire_texte(oeuvre["chemin"])
    doc = analyser_fr(texte)
    occ = [(s, e) for _, s, e in occurrences_fr(doc)]
    lignes = concordances_doc(doc, occ, oeuvre["oeuvre"], oeuvre["auteur"])
    nom = f"concordancier_{oeuvre['id']}_PER_FEM.json"
    exporter_json(lignes, nom)
    print(f"{oeuvre['id']:14} {len(lignes):5} mentions  →  {nom}")
    for ligne in lignes:
        ligne = dict(ligne)
        ligne["id_texte"] = oeuvre["id"]
        ligne["id"] = compteur
        compteur += 1
        concordancier_fr.append(ligne)

chemin_global = exporter_json(concordancier_fr, "concordancier_corpus_fr.json")
print(f"\nTotal français : {len(concordancier_fr)}  →  {chemin_global.name}")


AUCLERT          861 mentions  →  concordancier_AUCLERT_PER_FEM.json
NOIRFONTAINE     227 mentions  →  concordancier_NOIRFONTAINE_PER_FEM.json
DUCHESNE         461 mentions  →  concordancier_DUCHESNE_PER_FEM.json
CARTERON         312 mentions  →  concordancier_CARTERON_PER_FEM.json
CHELLIER         143 mentions  →  concordancier_CHELLIER_PER_FEM.json
BREMOND          612 mentions  →  concordancier_BREMOND_PER_FEM.json

Total français : 2616  →  concordancier_corpus_fr.json


### Concordanciers JSON — corpus arabe

Même procédure pour les **quatre** récits.


In [11]:
concordancier_ar = []
compteur = 1

for oeuvre in CORPUS_AR:
    texte = lire_texte(oeuvre["chemin"])
    lignes = concordances_ar(texte, oeuvre["oeuvre"], oeuvre["auteur"])
    nom = f"concordancier_{oeuvre['id']}_PER_FEM.json"
    exporter_json(lignes, nom)
    print(f"{oeuvre['id']:14} {len(lignes):5} mentions  →  {nom}")
    for ligne in lignes:
        ligne = dict(ligne)
        ligne["id_texte"] = oeuvre["id"]
        ligne["id"] = compteur
        compteur += 1
        concordancier_ar.append(ligne)

chemin_global = exporter_json(concordancier_ar, "concordancier_corpus_ar.json")
print(f"\nTotal arabe : {len(concordancier_ar)}  →  {chemin_global.name}")


SAADAWI          973 mentions  →  concordancier_SAADAWI_PER_FEM.json
SHIDIAQ          511 mentions  →  concordancier_SHIDIAQ_PER_FEM.json
MUBARAK          291 mentions  →  concordancier_MUBARAK_PER_FEM.json
BINNABI          289 mentions  →  concordancier_BINNABI_PER_FEM.json

Total arabe : 2064  →  concordancier_corpus_ar.json


### Option : relire les annotations Label Studio déjà faites

Les JSON arabes envoyés (AlSaadawi, Shidiaq, Mubarak, BinNabi) viennent de Label Studio. Cette cellule les convertit au même format de concordancier, pour comparer l’annotation manuelle et le lexique automatique.


In [12]:
def concordancier_depuis_labelstudio(chemin_json, oeuvre, auteur, texte_path=None):
    data = json.loads(Path(chemin_json).read_text(encoding="utf-8"))
    tache = data[0] if isinstance(data, list) else data
    anns = tache.get("annotations") or []
    if not anns:
        return []
    resultats = anns[0].get("result") or []
    texte = ""
    if texte_path:
        texte = lire_texte(texte_path)
    elif "text" in (tache.get("data") or {}):
        texte = tache["data"]["text"]

    tokens = tokeniser_ar(texte) if texte else []
    lignes = []
    for i, r in enumerate(resultats, start=1):
        v = r.get("value") or {}
        debut, fin = v.get("start"), v.get("end")
        mention = v.get("text") or (texte[debut:fin] if texte else "")
        item = {
            "id": i,
            "oeuvre": oeuvre,
            "auteur": auteur,
            "label": (v.get("labels") or ["PER FEM"])[0],
            "mention": mention,
            "start": debut,
            "end": fin,
        }
        if texte and tokens and debut is not None:
            i0 = next((k for k, t in enumerate(tokens) if t.idx >= debut), 0)
            i1 = next((k for k, t in enumerate(tokens) if t.idx >= fin), i0 + 1)
            c0, c1 = max(0, i0 - 10), min(len(tokens), i1 + 10)
            item["contexte_avant_10_tokens"] = "".join(t.text_with_ws for t in tokens[c0:i0])
            item["contexte_apres_10_tokens"] = "".join(t.text_with_ws for t in tokens[i1:c1])
            item["contexte_complet"] = "".join(t.text_with_ws for t in tokens[c0:c1])
        lignes.append(item)
    return lignes


# Fichiers Label Studio déposés à côté du notebook (si présents)
SOURCES_LS = {
    "SAADAWI": "AlSaadawi.json",
    "SHIDIAQ": "Shidiaq.json",
    "MUBARAK": "Mubarak.json",
    "BINNABI": "BinNabi.json",
}

for oeuvre in CORPUS_AR:
    nom_ls = SOURCES_LS.get(oeuvre["id"])
    if not nom_ls or not (DOSSIER_NOTEBOOK / nom_ls).exists():
        continue
    lignes = concordancier_depuis_labelstudio(
        DOSSIER_NOTEBOOK / nom_ls,
        oeuvre["oeuvre"],
        oeuvre["auteur"],
        oeuvre["chemin"],
    )
    nom = f"concordancier_{oeuvre['id']}_LabelStudio.json"
    exporter_json(lignes, nom)
    print(f"{oeuvre['id']:14} {len(lignes):5} annotations manuelles  →  {nom}")


SAADAWI          741 annotations manuelles  →  concordancier_SAADAWI_LabelStudio.json
SHIDIAQ          406 annotations manuelles  →  concordancier_SHIDIAQ_LabelStudio.json
MUBARAK          327 annotations manuelles  →  concordancier_MUBARAK_LabelStudio.json
BINNABI          305 annotations manuelles  →  concordancier_BINNABI_LabelStudio.json
